# 00 · NBA game video to 10 Hz court tracking

Runs the CourtVision pipeline on one broadcast clip and writes the dataset: **the court x, y of
every on-court player and the ball, ten times a second**.

    video ─► detect players, referees, ball, hoop (every frame)
          ─► court keypoints ─► image-to-court mapping, carried through pans by camera motion
          ─► track ids (camera-compensated, restarted at cuts) ─► teams by jersey colour
          ─► player court positions (relinked, smoothed) ─► ball court position
          ─► resample to 10 Hz ─► tracking.parquet

The models run on every video frame because tracking needs consecutive frames; only the output is
on the 10 Hz clock. Weights come from notebooks 01 and 02 (`MyDrive/courtvision/models/`).

In [ ]:
# Runtime > Change runtime type > GPU before running (A100 or L4 if you have them; T4 works, slower).
!nvidia-smi -L
from google.colab import drive, userdata
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/courtvision"   # everything that must survive a disconnect lives here
for d in ["models", "runs", "reports", "training_data", "input_videos", "output"]:
    os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)

In [ ]:
REPO_URL = "https://github.com/moorexavierb/CourtVision.git"
BRANCH = "main"   # to test a pull request, put its branch name here, then Runtime > Restart session > Run all
!rm -rf /content/courtvision && git clone -q -b {BRANCH} {REPO_URL} /content/courtvision
!pip -q install -e "/content/courtvision[models]"
import sys
sys.path.insert(0, "/content/courtvision")   # importable now, without restarting the session

## Settings

In [ ]:
from pathlib import Path
from courtvision import Config, pipeline

cfg = Config.load("/content/courtvision/configs/default.yaml")
cfg.models.detector = f"{DRIVE_ROOT}/models/player_detector.pt"
cfg.models.court_keypoints = f"{DRIVE_ROOT}/models/court_keypoints.pt"
cfg.output.dir = f"{DRIVE_ROOT}/output"
cfg.output.cache_dir = f"{DRIVE_ROOT}/stubs"   # model passes are cached per video, so re-runs skip the GPU
cfg.output.hz = 10                             # samples per second in tracking.parquet
cfg.output.team_names = ["Team A", "Team B"]   # display names; team 0 or 1 is decided by jersey colour

## Pick a clip (upload one, or put it in `MyDrive/courtvision/input_videos/`)

In [ ]:
import glob
videos = sorted(glob.glob(f"{DRIVE_ROOT}/input_videos/*.mp4"))
if not videos:
    from google.colab import files
    videos = list(files.upload())
VIDEO = videos[0]
print(VIDEO)

## Run

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
result = pipeline.run(VIDEO, cfg)
OUT = f"{cfg.output.dir}/{Path(VIDEO).stem}"
print("outputs in", OUT)

## Which stage is weak on this clip?

Read top to bottom: a stage can only be as good as the ones before it. If the court is mapped in
60% of frames, at most 60% of samples can have positions, however good the detector is.

In [ ]:
from courtvision.diagnostics import report
print(report(result.diagnostics))

In [ ]:
# Per frame: where the court mapping came from, players detected vs placed on the court, the ball.
import matplotlib.pyplot as plt
f = result.frames
fig, ax = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
ax[0].plot(f.frame, f.n_keypoints, lw=0.8); ax[0].set_ylabel("keypoints")
for src, c in [("keypoints", "tab:green"), ("fused", "tab:blue"), ("flow", "tab:orange"), ("none", "tab:red")]:
    m = f.court_source == src
    ax[0].scatter(f.frame[m], [-1] * m.sum(), s=2, c=c, label=src)
ax[0].legend(loc="upper right", fontsize=8, ncol=4)
ax[1].plot(f.frame, f.players_detected, lw=0.8, label="detected"); ax[1].plot(f.frame, f.players_placed, lw=0.8, label="placed")
ax[1].axhline(10, color="k", lw=0.5); ax[1].set_ylabel("players"); ax[1].legend(fontsize=8)
ax[2].plot(f.frame, f.ball_detected.astype(int), lw=0.8); ax[2].set_ylabel("ball seen"); ax[2].set_xlabel("frame")
for c in result.cuts:
    for a in ax:
        a.axvline(c, color="grey", lw=0.5, ls="--")
plt.tight_layout()

## The dataset: x, y for every player and the ball, ten times a second

`tracking.parquet` (and `.csv`), one row per sample per object. Court axes: `x_m` along the
sideline (0 to 28.65 m, baseline to baseline), `y_m` across (0 to 15.24 m).

| Column | |
|---|---|
| `sample`, `time_s`, `frame` | sample number, time on the 10 Hz clock, nearest video frame |
| `object`, `id`, `team` | `player` (player id, team) or `ball` (id -1) |
| `x_m`, `y_m`, `speed_mps` | court position in metres, speed in m/s (players) |
| `has_ball` | this player holds the ball |
| `source` | players: `tracked`. Ball: `held` (at the holder's hands), `flight` (straight ground line of a pass or shot), `floor` (loose ball, projected directly) |

Samples where an object was not seen for more than 0.2 s are left out rather than guessed.
Debugging tables next to it: `players_per_frame.parquet`, `ball_per_frame.parquet`, `frames.csv`.

In [ ]:
import pandas as pd
tracking = pd.read_parquet(f"{OUT}/tracking.parquet")
print(result.diagnostics["dataset"])
display(tracking.head(12))
# Players placed per sample: 10 means everyone on the court has a position.
tracking[tracking.object == "player"].groupby("sample").size().value_counts().sort_index()

In [ ]:
# Top-down replay of the first 20 seconds, one picture per sample.
import cv2
import numpy as np
from matplotlib import animation
from IPython.display import HTML
from courtvision.court import get_court
from courtvision.render import minimap
from courtvision.teams import team_bgr

court, colors = get_court(cfg.court.standard), team_bgr(result.team_colors)
t = result.tracking
samples = sorted(t["sample"].unique())[: int(20 * cfg.output.hz)]
fig, ax = plt.subplots(figsize=(10, 5.6)); ax.axis("off")
im = ax.imshow(np.zeros((10, 10, 3), np.uint8))

def draw(s):
    d = t[t["sample"] == s]
    b = d[d.object == "ball"]
    img = minimap(court, d[d.object == "player"], b[["x_m", "y_m"]].to_numpy()[0] if len(b) else None, colors, px_per_m=20)
    im.set_data(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); im.set_extent((0, img.shape[1], img.shape[0], 0))
    ax.set_title(f"t = {s / cfg.output.hz:.1f} s")
    return [im]

anim = animation.FuncAnimation(fig, draw, frames=samples, interval=1000 / cfg.output.hz)
plt.close(fig)
HTML(anim.to_jshtml())

## Annotated video (boxes, ids, team colours, ball, and the minimap in the corner)

In [ ]:
src = f"{OUT}/{Path(VIDEO).stem}_tracked.mp4"
!ffmpeg -y -loglevel error -i "{src}" -vcodec libx264 -crf 26 /content/preview.mp4
from IPython.display import Video
Video("/content/preview.mp4", embed=True, width=960)

## Download

In [ ]:
from google.colab import files
files.download(f"{OUT}/tracking.csv")